In [1]:
import pandas as pd
import numpy as np
import re
import string
import torch
from torch.autograd import Variable
import transformers

# Use torch's AdamW optimizer (transformers' AdamW is unavailable in this env)
from torch.optim import AdamW



In [2]:
train = pd.read_csv("../input/tweet-sentiment-extraction/train.csv")
test = pd.read_csv("../input/tweet-sentiment-extraction/test.csv")
sample_submission = pd.read_csv(
    "../input/tweet-sentiment-extraction/sample_submission.csv"
)



In [3]:
train.dropna(inplace=True)




In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub("https?://\S+|www\.\S+", "", text)
    text = re.sub("[%s]" % re.escape(string.punctuation), "", text)
    text = re.sub("\n", "", text)
    text = re.sub("\w*\d\w*", "", text)
    return text




In [5]:
train["text"] = train["text"].apply(lambda x: clean_text(x))
train["selected_text"] = train["selected_text"].apply(lambda x: clean_text(x))
test["text"] = test["text"].apply(lambda x: clean_text(x))



In [6]:
train_positive = train.loc[(train.sentiment == "positive")]
train_neutral = train.loc[(train.sentiment == "neutral")]
train_negative = train.loc[(train.sentiment == "negative")]
test_positive = test.loc[(test.sentiment == "positive")]
test_neutral = test.loc[(test.sentiment == "neutral")]
test_negative = test.loc[(test.sentiment == "negative")]



In [7]:
train_all = [train_positive, train_neutral, train_negative]
test_all = [test_positive, test_neutral, test_negative]




In [8]:
def jaccard(str1, str2):
    a = set(str1.lower().split())
    b = set(str2.lower().split())
    c = a.intersection(b)
    if (len(a) + len(b) - len(c)) == 0:
        return 1
    return float(len(c)) / (len(a) + len(b) - len(c))




In [9]:
def from_predicted_positon_to_text(predicted, threshold, padded_tokens, tokenizer):
    predicted = predicted.clone().cpu().numpy()
    predicted[predicted >= threshold] = 1
    predicted[predicted < threshold] = 0

    tokens_matrix = []
    decode_matrix = []
    for i in range(padded_tokens.shape[0]):
        decode = tokenizer.decode(padded_tokens[i])
        tokens_matrix.append(tokenizer.tokenize(decode))
        decode_matrix.append(" ".join(tokenizer.tokenize(decode)))

    index_matrix = []
    for i in range(len(predicted)):
        index = [i for i, p in enumerate(predicted[i, :].tolist()) if p == 1]
        index_matrix.append(index)

    first_last_words = []
    for i, j in zip(tokens_matrix, index_matrix):
        if 0 in j:
            j.remove(0)
        for k in j[::-1]:
            if i[k] in ["[SEP]", "[PAD]"]:
                j.remove(k)
        if len(j) == 0:
            first_last_words.append(["", ""])
        else:
            last = j[-1]
            first = j[0]
            if i[first] == i[last]:
                first_last_words.append(["", i[last]])
            else:
                first_last_words.append([i[first], i[last]])

    predicted_val_text = []
    for f_l, s in zip(first_last_words, decode_matrix):
        try:
            match = re.findall(f_l[0] + ".+" + f_l[1], s)[0]
            cleaned = (
                match.replace("[CLS]", "")
                .replace("CLS]", "")
                .replace("[SEP]", "")
                .replace("[PAD]", "")
                .replace("[S", "")
                .replace("[P", "")
                .replace("AD", "")
                .replace(" ##", "")
                .strip()
            )
            predicted_val_text.append(cleaned)
        except Exception:
            predicted_val_text.append("")
    return predicted_val_text




In [10]:
class BertModel(torch.nn.Module):
    def __init__(self, UNCASED, outputSize, droupout, std):
        super(BertModel, self).__init__()
        self.config = transformers.BertConfig.from_pretrained(
            UNCASED, output_hidden_states=True
        )
        self.bert_model = transformers.BertModel.from_pretrained(
            UNCASED, config=self.config
        )
        self.drop_out = torch.nn.Dropout(droupout)
        self.con_model1 = torch.nn.Conv1d(
            in_channels=768, out_channels=256, kernel_size=1
        )
        self.con_model2 = torch.nn.Conv1d(
            in_channels=256, out_channels=outputSize, kernel_size=1
        )
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        last_hidden_states = self.bert_model(
            input_ids, attention_mask=attention_mask.float()
        )
        last_hidden_states = last_hidden_states[0].permute(0, 2, 1)
        out = self.drop_out(last_hidden_states)
        out = self.con_model1(out)
        out = self.con_model2(out)
        out = torch.sum(out, dim=2)
        out = self.sigmoid(out)
        return out




In [11]:
torch.cuda.empty_cache()
category = ["positive", "neutral", "negative"]
color = ["b", "r", "g"]
max_sequence_length = 32
val_frac = 0.25
num_of_val = 1000
learningRate = 3e-5
max_length = max_sequence_length
outputSize = max_sequence_length
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = [3, 1, 4]
predicted_text = {}

# Use the public HuggingFace model identifier instead of a non‑existent local path
UNCASED = "bert-base-uncased"
std = 0.02
droupout = 0.1
batch_size = 16
no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]

for i in range(len(train_all)):
    threshold = 0 if i == 1 else 0.5
    tokenizer = transformers.BertTokenizer.from_pretrained(UNCASED)

    model = BertModel(UNCASED, outputSize, droupout, std).to(device)
    criterion = torch.nn.MSELoss()
    param_optimizer = list(model.named_parameters())
    optimizer_parameters = [
        {
            "params": [
                p for n, p in param_optimizer if not any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.001,
        },
        {
            "params": [
                p for n, p in param_optimizer if any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]
    optimizer = AdamW(optimizer_parameters, lr=learningRate)

    train_data = train_all[i]
    test_data = test_all[i]

    X_train_text = train_data.loc[:, "text"].tolist()
    y_train_text = train_data.loc[:, "selected_text"].tolist()
    X_test = test_data.loc[:, ["textID", "text"]]

    X_train_tokens = [
        tokenizer.encode(t, add_special_tokens=True, max_length=max_length)
        for t in X_train_text
    ]
    y_train_tokens = [
        tokenizer.encode(t, add_special_tokens=True, max_length=max_length)
        for t in y_train_text
    ]
    X_test_tokens = [
        tokenizer.encode(t, add_special_tokens=True, max_length=max_length)
        for t in X_test.text
    ]

    X_train_all_input_ids = np.array(
        [i + [0] * (max_length - len(i)) for i in X_train_tokens]
    )
    y_train_all_input_ids = np.array(
        [i + [0] * (max_length - len(i)) for i in y_train_tokens]
    )
    X_test_input_ids = np.array(
        [i + [0] * (max_length - len(i)) for i in X_test_tokens]
    )

    X_train_all_attention_mask = (X_train_all_input_ids != 0).astype(int)
    y_train_all_attention_mask = (y_train_all_input_ids != 0).astype(int)
    X_test_attention_mask = (X_test_input_ids != 0).astype(int)

    y_train_bool = []
    for j in range(len(y_train_all_input_ids)):
        a = [
            1 if x > 0 and x in y_train_all_input_ids[j, :] else 0
            for x in X_train_all_input_ids[j, :].tolist()
        ]
        y_train_bool.append(a)
    y_train_bool = np.array(y_train_bool)

    num_of_val = num_of_val + (len(X_train_all_input_ids) - num_of_val) % batch_size
    num_of_train = len(X_train_all_input_ids) - num_of_val

    X_val = X_train_all_input_ids[-num_of_val:, :]
    X_val_attention_mask = X_train_all_attention_mask[-num_of_val:, :]
    y_val = y_train_bool[-num_of_val:, :]

    training_loss = []

    model.train()
    for epoch in range(epochs[i]):
        for k_fold in range(int(num_of_train / batch_size)):
            start = k_fold * batch_size
            end = start + batch_size
            X_batch = X_train_all_input_ids[start:end, :]
            X_batch_mask = X_train_all_attention_mask[start:end, :]
            y_batch = y_train_bool[start:end, :]

            X_batch = Variable(torch.from_numpy(X_batch).to(device))
            X_batch_mask = Variable(torch.from_numpy(X_batch_mask).to(device))
            y_batch = Variable(torch.from_numpy(y_batch).to(device))

            outputs = model(X_batch, X_batch_mask)
            loss = criterion(outputs.float(), y_batch.float())
            training_loss.append(loss.item())
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            if k_fold % 100 == 0:
                print(
                    f"{i}-th category, fold {k_fold}, epoch {epoch}, loss {loss.item():.4f}"
                )

    model.eval()
    with torch.no_grad():
        X_val_v = Variable(torch.from_numpy(X_val).to(device))
        X_val_mask_v = Variable(torch.from_numpy(X_val_attention_mask).to(device))
        eval_predicted = model(X_val_v, X_val_mask_v)

    predicted_val_text = from_predicted_positon_to_text(
        eval_predicted, threshold, torch.tensor(X_val), tokenizer
    )
    jaccard_score_list = [
        jaccard(p, t) for p, t in zip(predicted_val_text, y_train_text[-num_of_val:])
    ]
    print(pd.Series(jaccard_score_list).describe())

    with torch.no_grad():
        X_test_v = Variable(torch.from_numpy(X_test_input_ids).to(device))
        X_test_mask_v = Variable(torch.from_numpy(X_test_attention_mask).to(device))
        test_predicted = model(X_test_v, X_test_mask_v)

    predicted_test_text = from_predicted_positon_to_text(
        test_predicted, threshold, torch.tensor(X_test_input_ids), tokenizer
    )
    for p, idx in zip(predicted_test_text, X_test.textID.tolist()):
        predicted_text[idx] = p



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

2026-05-02 13:07:50.467590: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777727270.477223      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777727270.479983      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-02 13:07:50.491319: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


...

0-th category, fold 0, epoch 0, loss 0.3741
0-th category, fold 100, epoch 0, loss 0.1596
0-th category, fold 200, epoch 0, loss 0.1166
0-th category, fold 300, epoch 0, loss 0.1262
0-th category, fold 400, epoch 0, loss 0.0718
0-th category, fold 0, epoch 1, loss 0.0793
0-th category, fold 100, epoch 1, loss 0.1145
0-th category, fold 200, epoch 1, loss 0.1054
0-th category, fold 300, epoch 1, loss 0.0948
0-th category, fold 400, epoch 1, loss 0.0505
0-th category, fold 0, epoch 2, loss 0.0637
0-th category, fold 100, epoch 2, loss 0.0994
0-th category, fold 200, epoch 2, loss 0.0820
0-th category, fold 300, epoch 2, loss 0.0860
0-th category, fold 400, epoch 2, loss 0.0441
count    1015.000000
mean        0.409200
std         0.319336
min         0.000000
25%         0.153846
50%         0.333333
75%         0.600000
max         1.000000
dtype: float64


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


1-th category, fold 0, epoch 0, loss 0.4367
1-th category, fold 100, epoch 0, loss 0.0504
1-th category, fold 200, epoch 0, loss 0.0331
1-th category, fold 300, epoch 0, loss 0.0523
1-th category, fold 400, epoch 0, loss 0.0276
1-th category, fold 500, epoch 0, loss 0.0213
count    1021.000000
mean        0.984802
std         0.070861
min         0.240000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
dtype: float64


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


2-th category, fold 0, epoch 0, loss 0.3358
2-th category, fold 100, epoch 0, loss 0.0979
2-th category, fold 200, epoch 0, loss 0.1338
2-th category, fold 300, epoch 0, loss 0.1693
2-th category, fold 0, epoch 1, loss 0.1332
2-th category, fold 100, epoch 1, loss 0.0781
2-th category, fold 200, epoch 1, loss 0.1044
2-th category, fold 300, epoch 1, loss 0.1437
2-th category, fold 0, epoch 2, loss 0.1107
2-th category, fold 100, epoch 2, loss 0.0633
2-th category, fold 200, epoch 2, loss 0.0691
2-th category, fold 300, epoch 2, loss 0.0945
2-th category, fold 0, epoch 3, loss 0.0617
2-th category, fold 100, epoch 3, loss 0.0429
2-th category, fold 200, epoch 3, loss 0.0569
2-th category, fold 300, epoch 3, loss 0.0537
count    1031.000000
mean        0.372137
std         0.328450
min         0.000000
25%         0.100000
50%         0.285714
75%         0.571429
max         1.000000
dtype: float64


In [12]:
submission_df = sample_submission.copy()
submission_df["selected_text"] = submission_df["textID"].apply(
    lambda x: predicted_text.get(x, "")
)
submission_df = submission_df[["textID", "selected_text"]]
submission_df.head()



,textID,selected_text
0,80a1e6bc32,my wish
1,863097735d,gosh today sucks i didnt get my tax returns im...
2,264cd5277f,tired and didnt really have an exciting saturd...
3,baee1e6ffc,ive been eating cheetos all morning
4,67d06a8dee,haiiii sankq im fineee ima js get a checkup co...


In [13]:
submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

Submission saved to submission.csv
